# Crop Production and Weather Analytics
**College Project - Google Colab**

**Dataset 1:** Kaggle `abhinand05/crop-production-in-india` - State, District, Year, Season, Crop, Area, Production (1997-2015).

**Dataset 2:** Kaggle `rajanand/rainfall-in-india` - IMD sub-division wise monthly rainfall with an `ANNUAL` column (1901-2015).

> Kaggle login: create an API token in Kaggle → Account → Create New API Token, then upload `kaggle.json` if `kagglehub` asks for it.

Run all cells from top to bottom.

## 1. Load and Inspect the Dataset

In [ ]:
# kagglehub downloads a Kaggle dataset straight into the Colab notebook
!pip install -q kagglehub pandas numpy matplotlib seaborn
import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

crop_folder = kagglehub.dataset_download('abhinand05/crop-production-in-india')
print('Crop dataset folder:', crop_folder)
print('Files:', os.listdir(crop_folder))

crop_csv = [f for f in os.listdir(crop_folder) if f.lower().endswith('.csv')][0]
df = pd.read_csv(os.path.join(crop_folder, crop_csv))
print('Loaded file:', crop_csv)

In [ ]:
print('SHAPE (rows, columns):', df.shape)
print('\nHEAD:')
display(df.head())
print('\nINFO:')
df.info()
print('\nDESCRIBE (all columns):')
display(df.describe(include='all'))

## 2. Data Cleaning

Steps: missing values → duplicates → text formatting → data types → invalid values → outliers.

In [ ]:
# Save a snapshot of the RAW data for the final before/after table
before = {
    'Rows': len(df),
    'Missing values': int(df.isna().sum().sum()),
    'Duplicate rows': int(df.duplicated().sum()),
    'Invalid rows': int(((df['Area'] <= 0) | (df['Production'] < 0)).sum())
}

print('Missing values per column:')
display(df.isna().sum().to_frame('Missing Count'))
print('Total missing values:', before['Missing values'])

In [ ]:
print('Rows before handling missing values:', len(df))
df = df.dropna()
print('Rows after dropping rows with missing values:', len(df))

In [ ]:
dup_count = int(df.duplicated().sum())
print('Duplicate rows found:', dup_count)
df = df.drop_duplicates().reset_index(drop=True)
print('Rows after removing duplicates:', len(df))

In [ ]:
text_cols = ['State_Name', 'District_Name', 'Season', 'Crop']
for col in text_cols:
    # remove leading/trailing spaces, collapse multiple spaces, standardise case
    df[col] = df[col].astype(str).str.strip().str.split().str.join(' ').str.title()

print('Unique text values after trimming spaces and fixing case:')
for col in text_cols:
    print(f'  {col}: {df[col].nunique()}')
display(df[text_cols].head())

In [ ]:
print('Data types BEFORE:')
print(df.dtypes)

# force numeric columns to numbers and the year to an integer
df['Crop_Year'] = pd.to_numeric(df['Crop_Year'], errors='coerce')
df['Area'] = pd.to_numeric(df['Area'], errors='coerce')
df['Production'] = pd.to_numeric(df['Production'], errors='coerce')
df = df.dropna(subset=['Crop_Year', 'Area', 'Production'])
df['Crop_Year'] = df['Crop_Year'].astype(int)

print('\nData types AFTER:')
print(df.dtypes)

In [ ]:
bad_area = df['Area'] <= 0
bad_production = df['Production'] < 0
print('Rows with Area <= 0:', int(bad_area.sum()))
print('Rows with Production < 0:', int(bad_production.sum()))
df = df[~(bad_area | bad_production)].reset_index(drop=True)
print('Rows after removing invalid values:', len(df))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=df['Area'], ax=axes[0], color='skyblue')
axes[0].set_title('Boxplot of Area (outlier check)')
axes[0].set_xlabel('Area (hectares)')
axes[0].set_ylabel('Distribution')
sns.boxplot(x=df['Production'], ax=axes[1], color='lightcoral')
axes[1].set_title('Boxplot of Production (outlier check)')
axes[1].set_xlabel('Production')
axes[1].set_ylabel('Distribution')
plt.tight_layout()
plt.show()

# IQR rule: a value is an outlier if it is below Q1-1.5*IQR or above Q3+1.5*IQR
for col in ['Area', 'Production']:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = int(((df[col] < low) | (df[col] > high)).sum())
    print(f'{col}: IQR fences = ({low:,.1f}, {high:,.1f}), outliers = {count}')

print('Note: high values are kept - large farms/harvests are real - so later charts use log scale.')

In [ ]:
after = {
    'Rows': len(df),
    'Missing values': int(df.isna().sum().sum()),
    'Duplicate rows': int(df.duplicated().sum()),
    'Invalid rows': int(((df['Area'] <= 0) | (df['Production'] < 0)).sum())
}
comparison = pd.DataFrame({'Before': before, 'After': after})
print('CLEANING SUMMARY')
display(comparison)

## 3. Add Yield = Production / Area

In [ ]:
df['Yield'] = df['Production'] / df['Area']
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=['Yield']).reset_index(drop=True)
print('Yield = Production / Area (tonnes per hectare)')
display(df[['State_Name', 'Crop', 'Area', 'Production', 'Yield']].head())

## 4. Merge a State-Level Annual Rainfall Dataset

Source: Kaggle `rajanand/rainfall-in-india` (IMD data, 1901-2015). We keep `SUBDIVISION`, `YEAR` and `ANNUAL` (total mm) and merge on state/sub-division + year.

In [ ]:
rain_folder = kagglehub.dataset_download('rajanand/rainfall-in-india')
print('Rainfall dataset folder:', rain_folder)
print('Files:', os.listdir(rain_folder))

rain_files = [f for f in os.listdir(rain_folder) if f.lower().endswith('.csv')]
rain_file = next((f for f in rain_files if '1901' in f and 'rainfall' in f.lower()), rain_files[0])
rain = pd.read_csv(os.path.join(rain_folder, rain_file))
print('Loaded:', rain_file, '| shape:', rain.shape)
display(rain.head())
print('Columns:', list(rain.columns))

In [ ]:
# Keep only the three columns we need and rename them
rain = rain[['SUBDIVISION', 'YEAR', 'ANNUAL']].copy()
rain.columns = ['Rain_State', 'Crop_Year', 'Rainfall']
rain['Rain_State'] = rain['Rain_State'].astype(str).str.strip().str.upper().str.split().str.join(' ')
rain['Crop_Year'] = pd.to_numeric(rain['Crop_Year'], errors='coerce')
rain['Rainfall'] = pd.to_numeric(rain['Rainfall'], errors='coerce')
rain = rain.dropna()
rain['Crop_Year'] = rain['Crop_Year'].astype(int)
rain = rain.groupby(['Rain_State', 'Crop_Year'], as_index=False)['Rainfall'].mean()
print('Clean rainfall table shape:', rain.shape)
display(rain.head())

In [ ]:
# IMD uses rainfall SUB-DIVISIONS, so some state names need renaming first
state_to_subdivision = {
    'ANDAMAN AND NICOBAR ISLANDS': 'ANDAMAN & NICOBAR ISLANDS',
    'JAMMU AND KASHMIR': 'JAMMU & KASHMIR',
    'DADRA AND NAGAR HAVELI': 'DADRA & NAGAR HAVELI',
    'ASSAM': 'ASSAM & MEGHALAYA',
    'MEGHALAYA': 'ASSAM & MEGHALAYA',
    'MANIPUR': 'NAGA MANI MIZO TRIPURA',
    'MIZORAM': 'NAGA MANI MIZO TRIPURA',
    'NAGALAND': 'NAGA MANI MIZO TRIPURA',
    'TRIPURA': 'NAGA MANI MIZO TRIPURA',
    'ODISHA': 'ORISSA',
    'KARNATAKA': 'COASTAL KARNATAKA',
    'CHANDIGARH': 'PUNJAB'
}
df['Rain_State'] = df['State_Name'].str.upper().replace(state_to_subdivision)
df = df.merge(rain, on=['Rain_State', 'Crop_Year'], how='left')

matched = df['Rainfall'].notna()
print('Total rows after merge:', len(df))
print('Matched with rainfall:', int(matched.sum()))
print('Unmatched rows:', int((~matched).sum()), f'({(~matched).mean() * 100:.1f}%)')
print('\nStates with unmatched rows:')
display(df.loc[~matched, 'State_Name'].value_counts().to_frame('Unmatched Rows'))

## 5. Univariate Analysis (one variable at a time)

In [ ]:
num_cols = ['Area', 'Production', 'Yield']
uni_stats = pd.DataFrame({
    'Mean': df[num_cols].mean(),
    'Median': df[num_cols].median(),
    'Mode': df[num_cols].mode().iloc[0],
    'Std': df[num_cols].std(),
    'Min': df[num_cols].min(),
    'Max': df[num_cols].max()
}).T
print('Central tendency and spread:')
display(uni_stats.round(2))

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(np.log10(df['Area'][df['Area'] > 0]), bins=50, color='steelblue', edgecolor='white')
plt.title('Histogram of Cultivated Area (log scale)')
plt.xlabel('log10(Area in hectares)')
plt.ylabel('Number of records')
plt.show()

plt.figure(figsize=(7, 4))
sns.boxplot(y=df['Yield'], color='lightcoral')
plt.title('Boxplot of Crop Yield')
plt.xlabel('Crop Yield')
plt.ylabel('Yield (tonnes per hectare)')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.countplot(data=df, x='Season', order=df['Season'].value_counts().index, ax=axes[0], color='seagreen')
axes[0].set_title('Number of Records by Season')
axes[0].set_xlabel('Season')
axes[0].set_ylabel('Number of records')

top10 = df['Crop'].value_counts().head(10).index
sns.countplot(data=df, x='Crop', order=top10, ax=axes[1], color='darkorange')
axes[1].set_title('Top 10 Crops by Number of Records')
axes[1].set_xlabel('Crop')
axes[1].set_ylabel('Number of records')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 6. Bivariate Analysis (two variables at a time)

In [ ]:
plot_df = df[(df['Area'] > 0) & (df['Production'] > 0)]
plt.figure(figsize=(7, 5))
plt.scatter(np.log10(plot_df['Area']), np.log10(plot_df['Production']), alpha=0.15, s=8, color='seagreen')
plt.title('Area vs Production (log-log scatter)')
plt.xlabel('log10(Area in hectares)')
plt.ylabel('log10(Production)')
plt.show()

In [ ]:
yearly = df.groupby('Crop_Year')['Production'].sum()
plt.figure(figsize=(8, 4))
plt.plot(yearly.index, yearly.values, marker='o', color='purple')
plt.title('Total Crop Production by Year')
plt.xlabel('Crop Year')
plt.ylabel('Total Production')
plt.xticks(yearly.index, rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
top5_states = df.groupby('State_Name')['Production'].sum().nlargest(5).index
state_season = df[df['State_Name'].isin(top5_states)].pivot_table(
    index='Season', columns='State_Name', values='Production', aggfunc='sum', fill_value=0
)
ax = state_season.plot(kind='bar', figsize=(11, 4))
ax.set_title('Production by Season for the Top 5 States')
ax.set_xlabel('Season')
ax.set_ylabel('Total Production')
ax.legend(title='State')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
top5_crops = df.groupby('Crop')['Production'].sum().nlargest(5).index
crop_df = df[df['Crop'].isin(top5_crops)]
plt.figure(figsize=(9, 4))
sns.boxplot(data=crop_df, x='Crop', y='Yield')
plt.ylim(0, crop_df['Yield'].quantile(0.99))
plt.title('Yield Distribution for the Top 5 Crops by Production')
plt.xlabel('Crop')
plt.ylabel('Yield (tonnes per hectare)')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
weather = df.dropna(subset=['Rainfall', 'Yield'])
x = weather['Rainfall']
y = weather['Yield']
slope, intercept = np.polyfit(x, y, 1)
trend_x = np.linspace(x.min(), x.max(), 100)

plt.figure(figsize=(7, 5))
plt.scatter(x, y, alpha=0.15, s=8, color='steelblue')
plt.plot(trend_x, slope * trend_x + intercept, color='red', linewidth=2,
         label=f'Trend: y = {slope:.4f}x + {intercept:.2f}')
plt.title('Annual Rainfall vs Crop Yield (with trend line)')
plt.xlabel('Annual Rainfall (mm)')
plt.ylabel('Yield (tonnes per hectare)')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Multivariate Analysis (three or more variables)

In [ ]:
corr_cols = ['Area', 'Production', 'Yield', 'Rainfall', 'Crop_Year']
corr = df[corr_cols].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap: Numeric Variables')
plt.xlabel('Variables')
plt.ylabel('Variables')
plt.tight_layout()
plt.show()

In [ ]:
top_states8 = df.groupby('State_Name')['Production'].sum().nlargest(8).index
state_year = df[df['State_Name'].isin(top_states8)].pivot_table(
    index='State_Name', columns='Crop_Year', values='Production', aggfunc='sum', fill_value=0
)
plt.figure(figsize=(12, 5))
sns.heatmap(state_year, cmap='YlGnBu')
plt.title('Production Heatmap: Top 8 States by Year')
plt.xlabel('Crop Year')
plt.ylabel('State')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=plot_df, x='Area', y='Production', hue='Season', alpha=0.5, s=15)
plt.xscale('log')
plt.yscale('log')
plt.title('Area vs Production Coloured by Season')
plt.xlabel('Area (hectares, log scale)')
plt.ylabel('Production (log scale)')
plt.legend(title='Season', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 8. Statistical Summary with describe()

In [ ]:
print('Final cleaned dataset shape:', df.shape)
print('\nDESCRIBE of numeric variables (including merged rainfall):')
display(df[['Area', 'Production', 'Yield', 'Rainfall', 'Crop_Year']].describe())

## 9. Final Rankings: Top and Bottom States, Crops and Seasons

In [ ]:
def show_rank(group_col, label):
    totals = df.groupby(group_col)['Production'].sum()
    print('=' * 60)
    print(f'TOP 5 {label} BY TOTAL PRODUCTION')
    print(totals.nlargest(5).to_string())
    print(f'\nBOTTOM 5 {label} BY TOTAL PRODUCTION')
    print(totals.nsmallest(5).to_string())
    print()

show_rank('State_Name', 'STATES')
show_rank('Crop', 'CROPS')
show_rank('Season', 'SEASONS')